In [0]:
data =[ 
    (101,'Gopal','IT',450000),
    (102,'Roshan','HR',55000),
    (103,'Nitin','Finance',65000),
    (104,'Dipak','Insurance',76000),
    (105,'Lalit','Production',56000)
]

columns = ['id','name','department','salary']
df = spark.createDataFrame(data,columns)
df.show()

+---+------+----------+------+
| id|  name|department|salary|
+---+------+----------+------+
|101| Gopal|        IT|450000|
|102|Roshan|        HR| 55000|
|103| Nitin|   Finance| 65000|
|104| Dipak| Insurance| 76000|
|105| Lalit|Production| 56000|
+---+------+----------+------+



Filter employees whose salary is greater than 50000.



Display only the name, department, and salary columns.



Store the final result in a variable named result.



Display the result.

In [0]:
from pyspark.sql.functions import *
df_fillterd = df.filter(col("salary")> 60000)
df_result = df_fillterd.select("name","department","salary")

result = df_result
result.show()


+-----+----------+------+
| name|department|salary|
+-----+----------+------+
|Gopal|        IT|450000|
|Nitin|   Finance| 65000|
|Dipak| Insurance| 76000|
+-----+----------+------+



Add a new column called incremented_salary.



Employees in the IT department receive a 10% increment.



All other employees receive a 5% increment.



Display name, department, salary, and incremented_salary.



Store the final DataFrame in result.

In [0]:
from pyspark.sql.functions import col , when
df1 = df.withColumn(
    "increment_salary",
when(
     col("department")=="IT",
     col("salary")*1.10
)    .otherwise(
        col("salary")*1.05
)
)
result = df1.select("name","department","salary","increment_salary")
result.explain("extended")
result.show()


== Parsed Logical Plan ==
'Project ['name, 'department, 'salary, 'increment_salary]
+- Project [id#10222L, name#10223, department#10224, salary#10225L, CASE WHEN (department#10224 = IT) THEN (cast(salary#10225L as double) * 1.1) ELSE (cast(salary#10225L as double) * 1.05) END AS increment_salary#10227]
   +- Project [id#10114L AS id#10222L, name#10115 AS name#10223, department#10116 AS department#10224, salary#10117L AS salary#10225L]
      +- LocalRelation [id#10114L, name#10115, department#10116, salary#10117L]

== Analyzed Logical Plan ==
name: string, department: string, salary: bigint, increment_salary: double
Project [name#10223, department#10224, salary#10225L, increment_salary#10227]
+- Project [id#10222L, name#10223, department#10224, salary#10225L, CASE WHEN (department#10224 = IT) THEN (cast(salary#10225L as double) * 1.1) ELSE (cast(salary#10225L as double) * 1.05) END AS increment_salary#10227]
   +- Project [id#10114L AS id#10222L, name#10115 AS name#10223, department#101

Group employees by department.



Calculate the average salary.



Count the employees in each department.



Name the output columns avg_salary and employee_count.



Store the final DataFrame in result and display it.

In [0]:
from pyspark.sql.functions import col , lit , avg ,count
result = df.groupBy("department").agg(
    avg(col("salary")).alias("avrage_salary"),
    count("*").alias('employee_count')
    )
           

result.show()




+----------+-------------+--------------+
|department|avrage_salary|employee_count|
+----------+-------------+--------------+
|        IT|     450000.0|             1|
|        HR|      55000.0|             1|
|   Finance|      65000.0|             1|
| Insurance|      76000.0|             1|
|Production|      56000.0|             1|
+----------+-------------+--------------+



In [0]:
data = [
    (1, "Pune", 2500),
    (2, "Mumbai", 4000),
    (3, "Pune", 3500),
    (4, "Nashik", 1500),
    (5, "Mumbai", 2000),
    (6, "Pune", 1800),
    (7, "Nashik", 4500),
    (8, "Mumbai", 1200)
]

columns = ["order_id", "city", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

+--------+------+------+
|order_id|  city|amount|
+--------+------+------+
|       1|  Pune|  2500|
|       2|Mumbai|  4000|
|       3|  Pune|  3500|
|       4|Nashik|  1500|
|       5|Mumbai|  2000|
|       6|  Pune|  1800|
|       7|Nashik|  4500|
|       8|Mumbai|  1200|
+--------+------+------+



Keep only orders where amount > 1000.



Group the remaining orders by city.



Calculate total sales per city as total_sales.



Keep only cities where total_sales > 5000.



Store the result in result and display it.

In [0]:
df1 = df.filter(col("amount")>1000)
df2 = df1.groupBy("city").agg(
    sum("amount").alias("total_sales")
)
df_result =df2.filter(col("total_sales")>5000
)
result = df_result
result.explain("extended")
result.show()








== Parsed Logical Plan ==
'Filter '`>`('total_sales, 5000)
+- 'Aggregate ['city], ['city, 'sum('amount) AS total_sales#10110]
   +- 'Filter '`>`('amount, 1000)
      +- 'UnresolvedSubqueryColumnAliases [order_id, city, amount]
         +- LocalRelation [order_id#10103L, city#10104, amount#10105L]

== Analyzed Logical Plan ==
city: string, total_sales: bigint
Filter (total_sales#10110L > cast(5000 as bigint))
+- Aggregate [city#10114], [city#10114, sum(amount#10115L) AS total_sales#10110L]
   +- Filter (amount#10115L > cast(1000 as bigint))
      +- Project [order_id#10103L AS order_id#10113L, city#10104 AS city#10114, amount#10105L AS amount#10115L]
         +- LocalRelation [order_id#10103L, city#10104, amount#10105L]

== Optimized Logical Plan ==
Filter (isnotnull(total_sales#10110L) AND (total_sales#10110L > 5000))
+- Aggregate [city#10114], [city#10114, sum(amount#10115L) AS total_sales#10110L]
   +- LocalRelation [city#10114, amount#10115L]

== Physical Plan ==
AdaptiveSparkPlan i

Group by city.



Calculate total_sales, order_count, and average_order_amount.



Keep cities with at least 2 orders and total sales greater than 5000.



Sort by total_sales in descending order.



Store the final result in result.

In [0]:

from pyspark.sql.functions import *
df1 = df.groupBy("city").agg(
    sum('amount').alias("total_sales"),
    avg("amount").alias("avrage_amount"),
    count("*").alias('total_order')
)
df2 = df1.filter(
(col("total_order")>=2)&
(col("total_sales")>5000)
)

result = df2.orderBy(
    col("total_sales").desc()
)
result.show()



+------+-----------+-------------+-----------+
|  city|total_sales|avrage_amount|total_order|
+------+-----------+-------------+-----------+
|  Pune|       7800|       2600.0|          3|
|Mumbai|       7200|       2400.0|          3|
|Nashik|       6000|       3000.0|          2|
+------+-----------+-------------+-----------+



In [0]:
data = [

    (1,"C101",2500,"2026-01-10"),
    (2,"C102",4000,"2026-01-12"),
    (3,"C103",3500,"2026-02-05"),
    (4, "C103", 1500, "2026-01-15"),
    (5, "C102", 2000, "2026-03-01"),
    (6, "C101", 1800, "2026-03-10"),
    (7, "C103", 4500, "2026-02-20"),
    (8, "C102", 1200, "2026-03-15")
]
columns = ["order_id","customer_id","amount","order_date"]
df = spark.createDataFrame(data,columns)
df.show()

+--------+-----------+------+----------+
|order_id|customer_id|amount|order_date|
+--------+-----------+------+----------+
|       1|       C101|  2500|2026-01-10|
|       2|       C102|  4000|2026-01-12|
|       3|       C103|  3500|2026-02-05|
|       4|       C103|  1500|2026-01-15|
|       5|       C102|  2000|2026-03-01|
|       6|       C101|  1800|2026-03-10|
|       7|       C103|  4500|2026-02-20|
|       8|       C102|  1200|2026-03-15|
+--------+-----------+------+----------+



Convert order_date to a date type.



Partition records by customer_id.



Order each customer's records by order_date descending.



Assign a row number to each order.



Keep only the latest order for each customer.



Display customer_id, order_id, amount, and order_date.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df1 = df.withColumn(
    "order_date",
    to_date(col("order_date"))
)
window_spece = Window.partitionBy(
    "customer_id",
).orderBy(
    col('order_date').desc()

)
df2 = df1.withColumn(
    "row_num",
    row_number().over(window_spece)
    
)
df3 = df2.filter(
    col("row_num")==1
)
result = df3.select("customer_id","amount","order_date")
result.show()


+-----------+------+----------+
|customer_id|amount|order_date|
+-----------+------+----------+
|       C101|  1800|2026-03-10|
|       C102|  1200|2026-03-15|
|       C103|  4500|2026-02-20|
+-----------+------+----------+



Create a window partitioned by department.



Rank salaries from highest to lowest using dense_rank().



Filter the results to keep only rank 2.



Display emp_id, name, department, and salary.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
window_Spect = Window.partitionBy(
    'department'
).orderBy(
    col('amount').desc()
) 
df2 = df.withColumn(
    "salary_rank",
    dense_rank().over(window_Spect)
)

result = df2.filter(
    col("salary_rank")==2
)
result.show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7366367298231148>, line 16
      8 df2 = df.withColumn(
      9     "salary_rank",
     10     dense_rank().over(window_Spect)
     11 )
     13 result = df2.filter(
     14     col("salary_rank")==2
     15 )
---> 16 result.show()

File <command-7366367298231148>, line 13
      3 window_Spect = Window.partitionBy(
      4     'department'
      5 ).orderBy(
      6     col('amount').desc()
      7 ) 
      8 df2 = df.withColumn(
      9     "salary_rank",
     10     dense_rank().over(window_Spect)
     11 )
---> 13 result = df2.filter(
     14     col("salary_rank")==2
     15 )
     16 result.show()

File <command-7366367298231148>, line 8
      2 from pyspark.sql.window import Window
      3 window_Spect = Window.partitionBy(
      4     'department'
      5 ).orderBy(
      6     col('amount').desc()
      7 ) 
----> 

Convert order_date into a date type.



Create a window partitioned by customer_id.



Order by order_date ascending.



Use sum() with .over() to calculate running_total.



Store the final DataFrame in result and display it.

In [0]:
from pyspark.sql .functions import *
from pyspark.sql.window import Window
df1 = df.withColumn(
    "order_date",
    to_date("order_date")
)
window_Spect = Window.partitionBy(
    "customer_id"
).orderBy(
        col("order_date").asc()
    ).rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
df2 = df1.withColumn(
    "running_total",
    sum('amount').over(window_Spect)
)
result = df2
result.show()




+--------+-----------+------+----------+-------------+
|order_id|customer_id|amount|order_date|running_total|
+--------+-----------+------+----------+-------------+
|       1|       C101|  2500|2026-01-10|         2500|
|       6|       C101|  1800|2026-03-10|         4300|
|       2|       C102|  4000|2026-01-12|         4000|
|       5|       C102|  2000|2026-03-01|         6000|
|       8|       C102|  1200|2026-03-15|         7200|
|       4|       C103|  1500|2026-01-15|         1500|
|       3|       C103|  3500|2026-02-05|         5000|
|       7|       C103|  4500|2026-02-20|         9500|
+--------+-----------+------+----------+-------------+

